# Lab: Multi-Layer Perceptron (MLP)

## 1. Từ Perceptron đến MLP

**Perceptron** (1958) là neuron đơn giản nhất:
$$\hat{y} = \text{sign}(w^T x + b)$$

Nó chỉ học được ranh giới *tuyến tính*. Không thể giải bài XOR (1969 — Minsky & Papert chỉ ra).

**Giải pháp:** xếp chồng nhiều perceptron, thêm hàm phi tuyến giữa các tầng → **Multi-Layer Perceptron**.
$$h_1 = \sigma(W_1 x + b_1)$$
$$h_2 = \sigma(W_2 h_1 + b_2)$$
$$\hat{y} = \text{softmax}(W_3 h_2 + b_3)$$

Theo **Universal Approximation Theorem**, MLP với 1 tầng ẩn đủ rộng + hàm kích hoạt phi tuyến có thể xấp xỉ *bất kỳ* hàm liên tục.

## 2. Hàm kích hoạt

- **ReLU** $\phi(z) = \max(0, z)$: nhanh, gradient ổn, default cho hidden layers.
- **Sigmoid** $\phi(z) = 1/(1+e^{-z})$: $\in (0,1)$, dùng cho output binary.
- **Softmax** $\phi(z)_i = e^{z_i}/\sum_j e^{z_j}$: dùng cho output multiclass.
- **Tanh** $\phi(z) \in (-1,1)$: ít dùng giờ nhưng vẫn ổn.

## 3. Loss và Backpropagation

Với phân loại nhiều lớp (Cross-Entropy Loss):
$$L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{c=1}^{C} y_{i,c} \log \hat{y}_{i,c}$$

Train bằng **backprop**: tính gradient của $L$ theo từng tham số bằng chain rule, đi ngược từ output về input. Cập nhật tham số bằng gradient descent (hoặc Adam, SGD+momentum...).

**Bẫy phổ biến**: PyTorch's `nn.CrossEntropyLoss` đã *tự* áp dụng softmax bên trong. Nên model trả về **logits**, KHÔNG đặt softmax cuối. Nếu đặt cả hai → softmax(softmax(z)) → loss bị méo, train chậm.

# THỰC HÀNH: MLP phân loại FashionMNIST

FashionMNIST: 70.000 ảnh 28×28 trắng đen của 10 loại đồ thời trang (áo, quần, giày, túi...). Khó hơn MNIST một chút.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])
trainset = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
testset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(testset,  batch_size=128, shuffle=False, num_workers=0)

class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
print(f'Train: {len(trainset)},  Test: {len(testset)}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.flat, np.random.choice(len(trainset), 10, replace=False)):
    img, label = trainset[i]
    ax.imshow(img.squeeze() * 0.3530 + 0.2860, cmap='gray')
    ax.set_title(class_names[label]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128),     nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters: {n_params:,}')

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.size(0)
    return loss_sum / total, correct / total

num_epochs = 10
train_loss_h, train_acc_h, test_loss_h, test_acc_h = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    running, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total   += y.size(0)

    tr_loss, tr_acc = running / total, correct / total
    te_loss, te_acc = evaluate(model, test_loader)
    train_loss_h.append(tr_loss); train_acc_h.append(tr_acc)
    test_loss_h.append(te_loss);  test_acc_h.append(te_acc)
    print(f'Epoch {epoch+1:2d}/{num_epochs}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc*100:.2f}%  '
          f'test_loss={te_loss:.4f}  test_acc={te_acc*100:.2f}%')

In [ ]:
epochs = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, train_loss_h, 'o-', label='Train')
axes[0].plot(epochs, test_loss_h,  's-', label='Test')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [a*100 for a in train_acc_h], 'o-', label='Train')
axes[1].plot(epochs, [a*100 for a in test_acc_h],  's-', label='Test')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
model.eval()
cm = torch.zeros(10, 10, dtype=torch.long)
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x.to(device)).argmax(1).cpu()
        for t, p in zip(y, preds):
            cm[t.item(), p.item()] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm.numpy(), cmap='Blues')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix — MLP trên FashionMNIST')
for i in range(10):
    for j in range(10):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

off_diag = cm.clone(); off_diag.fill_diagonal_(0)
i_max, j_max = np.unravel_index(off_diag.numpy().argmax(), off_diag.shape)
print(f'Cặp nhầm nhiều nhất: "{class_names[i_max]}" → "{class_names[j_max]}" '
      f'({cm[i_max, j_max]} lần) — hai loại trông giống nhau.')

## Tổng kết

1. MLP = nhiều tầng Linear + phi tuyến (ReLU). Học được ranh giới phi tuyến tuỳ ý nếu đủ lớn.
2. **Output là logits**, KHÔNG softmax cuối khi dùng `CrossEntropyLoss`.
3. Loss xuống, accuracy lên — nếu test loss bắt đầu tăng trong khi train loss vẫn giảm → **overfitting**.
4. **Confusion matrix** chỉ ra cặp lớp dễ nhầm — gợi ý cải tiến model hoặc dữ liệu.

## Khi nào MLP đủ — khi nào cần CNN?
MLP "flatten" ảnh thành vector → vứt mất thông tin không gian (pixel cạnh nhau). Với ảnh, **CNN** thường vượt MLP với ít tham số hơn nhiều.

# BÀI TẬP VỀ NHÀ

## Bài 1: Sâu hơn có tốt hơn không?
Thử 3 kiến trúc và so sánh test acc sau 10 epoch:
1. `784 → 128 → 10`
2. `784 → 256 → 128 → 10` (như lab)
3. `784 → 512 → 256 → 128 → 10`

Có hiện tượng diminishing returns (thêm tầng không giúp nhiều) không?

## Bài 2: Dropout chống overfitting
Thêm `nn.Dropout(0.3)` sau mỗi `nn.ReLU()`. Train 20 epoch. So sánh khoảng cách `train_acc - test_acc` trước và sau khi thêm Dropout. Dropout có giúp giảm gap không?

*Gợi ý:* Dropout chỉ active khi `model.train()`.

## Bài 3: Optimizer comparison
Train cùng kiến trúc với 3 optimizer:
- `optim.SGD(lr=0.01)`
- `optim.SGD(lr=0.01, momentum=0.9)`
- `optim.Adam(lr=1e-3)` (hiện tại)

Vẽ 3 đường loss trên cùng đồ thị. Cái nào hội tụ nhanh nhất?

## Bài 4: Áp dụng cho MNIST
Đổi `datasets.FashionMNIST` thành `datasets.MNIST`. Đổi normalize mean/std (`(0.1307,), (0.3081,)`). Train 10 epoch. So với FashionMNIST: MNIST dễ hơn (>97%) hay khó hơn?

## Bài 5: CIFAR-10 (nâng cao)
CIFAR-10: ảnh màu 32×32, 10 lớp đối tượng.
1. `datasets.CIFAR10(root='./data', ...)`.
2. Mean/std: `(0.4914, 0.4822, 0.4465)`, `(0.2470, 0.2435, 0.2616)`.
3. Input dim = 3 × 32 × 32 = 3072.
4. Train 15 epoch.

Quan sát: MLP trên CIFAR-10 thường chỉ đạt ~50% — kém xa CNN (~85%+). Vì sao? *MLP flatten ảnh, mất thông tin không gian. CNN dùng convolutional filter → khai thác được spatial structure.*